# 04. Combined uncertainty analysis

Combine evidence- and answer-stage uncertainty and evaluate whether they identify incorrect representative answers.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import requests
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

## 1. Configuration

In [ ]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd] + list(cwd.parents):
        if (path / "outputs").exists():
            return path
    return cwd

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"

SAMPLE_MODE = True
RUN_NAME = "sample" if SAMPLE_MODE else "full"

EVIDENCE_DIR = OUTPUT_DIR / "evidence_selection" / RUN_NAME
ANSWER_DIR = OUTPUT_DIR / "answer_generation" / RUN_NAME
ANALYSIS_DIR = OUTPUT_DIR / "combined_analysis" / RUN_NAME
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_BACKEND = "ollama" if SAMPLE_MODE else "vllm"

OLLAMA_JUDGE_MODEL = "gemma3:12b"
VLLM_JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

JUDGE_MODEL = (
    OLLAMA_JUDGE_MODEL
    if JUDGE_BACKEND == "ollama"
    else VLLM_JUDGE_MODEL
)

VLLM_BASE_URL = "http://localhost:8000/v1"

JUDGE_TEMPERATURE = 0.0
JUDGE_NUM_CTX = 8192

JUDGE_PATH = ANALYSIS_DIR / "judge_results.jsonl"
CASE_PATH = ANALYSIS_DIR / "case_analysis.csv"
CORRELATION_PATH = ANALYSIS_DIR / "correlations.csv"
PROFILE_PATH = ANALYSIS_DIR / "joint_profile_summary.csv"
PREDICTION_PATH = ANALYSIS_DIR / "prediction_metrics.csv"
RISK_PATH = ANALYSIS_DIR / "risk_coverage.csv"
SELECTIVE_PATH = ANALYSIS_DIR / "selective_review.csv"
HELDOUT_DIR = OUTPUT_DIR / "archehr_heldout"
HELDOUT_PATH = HELDOUT_DIR / "case_analysis.csv"


RUN_BOOTSTRAP = not SAMPLE_MODE
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
BOOTSTRAP_ALPHA = 0.05

BOOTSTRAP_METRICS_PATH = ANALYSIS_DIR / "bootstrap_metrics.csv"
BOOTSTRAP_COMPARISONS_PATH = (
    ANALYSIS_DIR / "bootstrap_comparisons.csv"
)

## 2. Shared functions

In [ ]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def format_references(reference_answers):
    return "\n".join(
        f"Reference {i}: {answer}"
        for i, answer in enumerate(reference_answers, 1)
    )

def build_judge_prompt(question, reference_answers, answer):
    return f'''
Evaluate the correctness of the generated biomedical answer.

Question:
{question}

Expert reference answer(s):
{format_references(reference_answers)}

Generated answer:
{answer}

Mark the generated answer as correct only if it is substantively correct
and sufficiently complete to answer the question.

Do not penalise paraphrasing or concise wording.
For a multi-part question, omitting a major requested part is incorrect.
A major contradiction or unsupported claim is incorrect.
The generated answer does not need to match every reference exactly.

Return only JSON:
{{"correct": true, "reason": "brief reason"}}
'''.strip()

def call_ollama(prompt, temperature=0.0):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": JUDGE_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p": 0.9,
                "num_predict": 128,
                "num_ctx": JUDGE_NUM_CTX,
            },
        },
        timeout=600,
    )
    response.raise_for_status()

    return response.json()["response"].strip()


def call_vllm(prompt, temperature=0.0):
    response = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": JUDGE_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": temperature,
            "top_p": 0.9,
            "max_completion_tokens": 128,
            "stream": False,
        },
        timeout=600,
    )
    response.raise_for_status()

    return (
        response.json()["choices"][0]["message"]["content"]
        .strip()
    )


def call_judge_model(prompt, temperature=0.0):
    if JUDGE_BACKEND == "ollama":
        return call_ollama(
            prompt,
            temperature=temperature,
        )

    if JUDGE_BACKEND == "vllm":
        return call_vllm(
            prompt,
            temperature=temperature,
        )

    raise ValueError(
        f"Unknown judge backend: {JUDGE_BACKEND}"
    )


def check_judge_server():
    if JUDGE_BACKEND == "ollama":
        url = "http://localhost:11434/"
    else:
        url = f"{VLLM_BASE_URL}/models"

    response = requests.get(
        url,
        timeout=10,
    )
    response.raise_for_status()


def parse_judgement(text):
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
        if match is None:
            return None
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    correct = data.get("correct")
    if isinstance(correct, bool):
        value = correct
    elif isinstance(correct, int) and correct in (0, 1):
        value = bool(correct)
    else:
        return None

    return {
        "correct": value,
        "reason": str(data.get("reason", "")).strip(),
    }

def judge_answer(question, reference_answers, answer):
    prompt = build_judge_prompt(
        question,
        reference_answers,
        answer,
    )

    for _ in range(2):
        result = parse_judgement(
            call_judge_model(
                prompt,
                temperature=JUDGE_TEMPERATURE,
            )
        )
        if result is not None:
            return result

    raise ValueError("Could not parse judge response.")

def representative_answer(group):
    group = group.sort_values("run_id").copy()
    group["cluster_label"] = group["cluster_label"].astype(int)

    counts = group["cluster_label"].value_counts()
    largest = counts.max()
    tied_labels = set(counts[counts == largest].index.tolist())

    # If clusters tie, use the cluster containing the earliest run.
    dominant_label = int(
        group[group["cluster_label"].isin(tied_labels)]
        .iloc[0]["cluster_label"]
    )

    dominant = group[
        group["cluster_label"] == dominant_label
    ].sort_values("run_id")

    row = dominant.iloc[0]

    return {
        "representative_answer": row["answer"],
        "representative_run_id": int(row["run_id"]),
        "representative_cluster_size": len(dominant),
    }

def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).dropna()

    if (
        len(data) < 3
        or data["x"].nunique() < 2
        or data["y"].nunique() < 2
    ):
        return np.nan, np.nan, len(data)

    rho, p_value = spearmanr(data["x"], data["y"])
    return float(rho), float(p_value), len(data)

def safe_auroc(y_true, scores):
    data = pd.DataFrame({
        "y": y_true,
        "score": scores,
    }).dropna()

    if len(data) < 2 or data["y"].nunique() < 2:
        return np.nan

    return float(
        roc_auc_score(
            data["y"].astype(int),
            data["score"],
        )
    )

# Calculate risk-coverage curve with tie handling
def compute_risk_coverage(group, signal):

    data = (
        group[[signal, "answer_error"]]
        .dropna()
        .copy()
    )

    # Group cases with the same uncertainty score
    tie_groups = (
        data.groupby(signal, sort=True)["answer_error"]
        .agg(["size", "sum"])
        .reset_index()
    )

    rows = []

    retained_before = 0
    errors_before = 0.0
    n = len(data)

    for row in tie_groups.itertuples(index=False):

        tie_n = row.size
        tie_errors = row.sum
        tie_error_rate = tie_errors / tie_n

        # Use expected error within each tied group
        for r in range(1, tie_n + 1):

            n_retained = retained_before + r

            expected_errors = (
                errors_before
                + r * tie_error_rate
            )

            error_rate = (
                expected_errors / n_retained
            )

            rows.append({
                "coverage": n_retained / n,
                "n_retained": n_retained,
                "retained_error_rate": error_rate,
                "retained_correctness": 1.0 - error_rate,
            })

        retained_before += tie_n
        errors_before += tie_errors

    return pd.DataFrame(rows)


# Report risk at selected coverage levels
def selective_review(
    group,
    signal,
    targets=(1.0, 0.9, 0.8, 0.7, 0.6),
):

    curve = compute_risk_coverage(
        group,
        signal,
    )

    rows = []
    n = len(curve)

    for target in targets:

        k = max(
            1,
            int(np.ceil(target * n)),
        )

        row = curve.iloc[k - 1]

        rows.append({
            "target_coverage": target,
            "coverage": row["coverage"],
            "n_retained": int(row["n_retained"]),
            "retained_error_rate": row["retained_error_rate"],
            "retained_correctness": row["retained_correctness"],
        })

    return pd.DataFrame(rows)

## 3. Load previous outputs

In [ ]:
evidence_summary = pd.read_csv(
    EVIDENCE_DIR / "evidence_summary.csv"
)

answer_summary = pd.read_csv(
    ANSWER_DIR / "answer_summary.csv"
)

answer_runs = pd.read_csv(
    ANSWER_DIR / "answer_runs_with_clusters.csv"
)

analysis = evidence_summary.merge(
    answer_summary,
    on=["analysis_set", "case_id"],
    how="inner",
    validate="one_to_one",
)

current_keys = set(
    zip(
        analysis["analysis_set"],
        analysis["case_id"],
    )
)

answer_runs = answer_runs[
    [
        (analysis_set, case_id) in current_keys
        for analysis_set, case_id in zip(
            answer_runs["analysis_set"],
            answer_runs["case_id"],
        )
    ]
].copy()

case_files = {
    "bioasq_train": PROCESSED_DIR / "bioasq_train_cases.jsonl",
    "bioasq_test": PROCESSED_DIR / "bioasq_test_cases.jsonl",
    "archehr_train": PROCESSED_DIR / "archehr_train_cases.jsonl",
    "archehr_test": PROCESSED_DIR / "archehr_test_cases.jsonl",
}

active_sets = set(analysis["analysis_set"])

case_by_key = {
    (analysis_set, case["case_id"]): case
    for analysis_set, path in case_files.items()
    if analysis_set in active_sets
    for case in load_jsonl(path)
}

print("Cases loaded:", len(analysis))
print(analysis["analysis_set"].value_counts())

## 4. Select representative answers

The representative answer is taken from the largest semantic answer cluster from Notebook 03.

In [ ]:
representative_rows = []

for (analysis_set, case_id), group in answer_runs.groupby(
    ["analysis_set", "case_id"],
    sort=False,
):
    row = representative_answer(group)
    row.update({
        "analysis_set": analysis_set,
        "case_id": case_id,
    })
    representative_rows.append(row)

representative_df = pd.DataFrame(representative_rows)

analysis = analysis.merge(
    representative_df,
    on=["analysis_set", "case_id"],
    how="left",
    validate="one_to_one",
)

if analysis["representative_answer"].isna().any():
    missing = analysis.loc[
        analysis["representative_answer"].isna(),
        ["analysis_set", "case_id"],
    ]
    raise ValueError(
        f"Missing representative answers:\n{missing}"
    )

display(
    analysis[
        [
            "analysis_set",
            "case_id",
            "evidence_uncertainty",
            "answer_uncertainty",
            "answer_pairwise_distance",
            "representative_cluster_size",
        ]
    ]
)

## 5. Judge representative-answer correctness

Correctness is evaluated automatically against the expert reference answer(s). This is an evaluation proxy rather than clinical ground truth.

In [ ]:
existing_judges = load_jsonl(JUDGE_PATH)

judge_lookup = {
    (
        row["analysis_set"],
        row["case_id"],
        row["representative_answer"],
    ): row
    for row in existing_judges
}

# Check whether any judge results are missing
missing_judges = [
    row
    for row in analysis.itertuples()
    if (
        row.analysis_set,
        row.case_id,
        row.representative_answer,
    ) not in judge_lookup
]

print("Loaded judge results:", len(judge_lookup))
print("Missing judge results:", len(missing_judges))

# Only run the judge model if results are missing
if missing_judges:

    try:
        check_judge_server()
    except Exception as exc:
        raise RuntimeError(
            f"Start {JUDGE_BACKEND} before running answer evaluation."
        ) from exc

    for row in missing_judges:

        case = case_by_key[
            (row.analysis_set, row.case_id)
        ]

        result = judge_answer(
            case["question"],
            case["reference_answers"],
            row.representative_answer,
        )

        record = {
            "analysis_set": row.analysis_set,
            "case_id": row.case_id,
            "representative_answer": row.representative_answer,
            "correct": bool(result["correct"]),
            "reason": result["reason"],
        }

        append_jsonl(record, JUDGE_PATH)

        key = (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )

        judge_lookup[key] = record

        print(
            row.analysis_set,
            row.case_id,
            result["correct"],
        )


analysis["judge_correct"] = [
    judge_lookup[
        (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )
    ]["correct"]
    for row in analysis.itertuples()
]

analysis["judge_reason"] = [
    judge_lookup[
        (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )
    ]["reason"]
    for row in analysis.itertuples()
]

analysis["judge_correct"] = (
    analysis["judge_correct"].astype(int)
)

analysis["answer_error"] = (
    1 - analysis["judge_correct"]
)

## 6. Combined analysis

In [ ]:
# Both primary uncertainty measures are already normalised to 0-1.
analysis["combined_uncertainty"] = (
    analysis["evidence_uncertainty"]
    + analysis["answer_uncertainty"]
) / 2.0


# Joint behavioural profiles
# EU = 0: stable evidence selection
# EU > 0: unstable evidence selection
# AU = 0: stable answer generation
# AU > 0: variable answer generation

analysis["evidence_profile"] = np.where(
    analysis["evidence_uncertainty"] == 0,
    "Stable",
    "Unstable",
)

analysis["answer_profile"] = np.where(
    analysis["answer_uncertainty"] == 0,
    "Stable",
    "Variable",
)

analysis["joint_profile"] = (
    analysis["evidence_profile"]
    + "–"
    + analysis["answer_profile"]
)


# Save case-level analysis with profile labels
analysis.to_csv(
    CASE_PATH,
    index=False,
)


# Correlations
correlation_rows = []

correlation_pairs = [
    (
        "evidence_vs_answer_uncertainty",
        "evidence_uncertainty",
        "answer_uncertainty",
    ),
    (
        "evidence_uncertainty_vs_answer_error",
        "evidence_uncertainty",
        "answer_error",
    ),
    (
        "answer_uncertainty_vs_answer_error",
        "answer_uncertainty",
        "answer_error",
    ),
    (
        "combined_uncertainty_vs_answer_error",
        "combined_uncertainty",
        "answer_error",
    ),
    (
        "pairwise_distance_vs_answer_error",
        "answer_pairwise_distance",
        "answer_error",
    ),
    (
        "evidence_uncertainty_vs_evidence_recall",
        "evidence_uncertainty",
        "evidence_recall",
    ),
    (
        "evidence_uncertainty_vs_evidence_f1",
        "evidence_uncertainty",
        "evidence_f1",
    ),
]

for analysis_set, group in analysis.groupby(
    "analysis_set",
    sort=False,
):
    for relationship, x_col, y_col in correlation_pairs:
        rho, p_value, n = safe_spearman(
            group[x_col],
            group[y_col],
        )

        correlation_rows.append({
            "analysis_set": analysis_set,
            "relationship": relationship,
            "n": n,
            "spearman_rho": rho,
            "p_value": p_value,
        })

correlations = pd.DataFrame(correlation_rows)

correlations.to_csv(
    CORRELATION_PATH,
    index=False,
)


# Reliability prediction
primary_signals = [
    "evidence_uncertainty",
    "answer_uncertainty",
    "combined_uncertainty",
]

companion_signals = [
    "answer_pairwise_distance",
]

prediction_rows = []
risk_rows = []
selective_rows = []

for analysis_set, group in analysis.groupby(
    "analysis_set",
    sort=False,
):
    for signal in primary_signals + companion_signals:

        curve = compute_risk_coverage(
            group,
            signal,
        )

        for row in curve.itertuples():
            risk_rows.append({
                "analysis_set": analysis_set,
                "signal": signal,
                "coverage": row.coverage,
                "n_retained": row.n_retained,
                "retained_error_rate": row.retained_error_rate,
                "retained_correctness": row.retained_correctness,
            })

        prediction_rows.append({
            "analysis_set": analysis_set,
            "signal": signal,
            "signal_role": (
                "primary"
                if signal in primary_signals
                else "companion"
            ),
            "n_cases": len(group),
            "n_incorrect": int(
                group["answer_error"].sum()
            ),
            "incorrect_rate": (
                group["answer_error"].mean()
            ),
            "auroc": safe_auroc(
                group["answer_error"],
                group[signal],
            ),
            # Lower AURC is better.
            "aurc": (
                curve["retained_error_rate"].mean()
            ),
        })

    for signal in primary_signals:
        table = selective_review(
            group,
            signal,
        )

        for row in table.itertuples():
            selective_rows.append({
                "analysis_set": analysis_set,
                "signal": signal,
                "target_coverage": row.target_coverage,
                "coverage": row.coverage,
                "n_retained": row.n_retained,
                "retained_error_rate": row.retained_error_rate,
                "retained_correctness": row.retained_correctness,
            })


prediction_metrics = pd.DataFrame(
    prediction_rows
)

risk_coverage = pd.DataFrame(
    risk_rows
)

selective_review_table = pd.DataFrame(
    selective_rows
)

prediction_metrics.to_csv(
    PREDICTION_PATH,
    index=False,
)

risk_coverage.to_csv(
    RISK_PATH,
    index=False,
)

selective_review_table.to_csv(
    SELECTIVE_PATH,
    index=False,
)

# Joint profile summary
profile_order = [
    "Stable–Stable",
    "Stable–Variable",
    "Unstable–Stable",
    "Unstable–Variable",
]

analysis["joint_profile"] = pd.Categorical(
    analysis["joint_profile"],
    categories=profile_order,
    ordered=True,
)

profile_summary = (
    analysis
    .groupby(
        ["analysis_set", "joint_profile"],
        observed=False,
    )
    .agg(
        n=("case_id", "size"),
        correct_n=("judge_correct", "sum"),
        incorrect_n=("answer_error", "sum"),
        error_rate=("answer_error", "mean"),
        median_eu=("evidence_uncertainty", "median"),
        median_au=("answer_uncertainty", "median"),
    )
    .reset_index()
)

profile_summary["profile_percent"] = (
    profile_summary["n"]
    / profile_summary.groupby(
        "analysis_set"
    )["n"].transform("sum")
    * 100
)

profile_summary.to_csv(
    PROFILE_PATH,
    index=False,
)


print("Saved:", ANALYSIS_DIR)

display(
    analysis.groupby("analysis_set")[
        [
            "evidence_uncertainty",
            "answer_uncertainty",
            "answer_pairwise_distance",
            "judge_correct",
        ]
    ].mean()
)

display(
    prediction_metrics[
        prediction_metrics["signal_role"]
        == "primary"
    ]
)

display(
    correlations[
        correlations["relationship"]
        == "evidence_vs_answer_uncertainty"
    ]
)

display(
    selective_review_table[
        selective_review_table["signal"]
        == "combined_uncertainty"
    ]
)

display(profile_summary)

## 7. Bootstrap 95% confidence intervals

Bootstrap 95% confidence intervals were used for the final RQ3 and RQ4 metrics.

In [ ]:
# Calculate percentile-based 95% CI
def percentile_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    return (
        np.quantile(values, alpha / 2),
        np.quantile(values, 1 - alpha / 2),
    )

# Calculate AURC from the risk-coverage curve
def aurc_score(df, signal):
    curve = compute_risk_coverage(df, signal)
    return curve["retained_error_rate"].mean()

# Resample correct and incorrect cases separately
def stratified_sample(df, rng):
    sampled = []

    for label in df["answer_error"].unique():
        idx = df.index[df["answer_error"] == label]
        sampled.extend(
            rng.choice(idx, size=len(idx), replace=True)
        )

    return df.loc[sampled].reset_index(drop=True)

# Bootstrap 95% CI for RQ3 Spearman correlation
def bootstrap_spearman(
    df,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):     

    rng = np.random.default_rng(seed)

    rho, _, _ = safe_spearman(
        df["evidence_uncertainty"],
        df["answer_uncertainty"],
    )

    boot_rho = []

    for _ in range(n_bootstrap):
        sample = df.sample(
            n=len(df),
            replace=True,
            random_state=int(rng.integers(0, 1_000_000)),
        )

        r, _, _ = safe_spearman(
            sample["evidence_uncertainty"],
            sample["answer_uncertainty"],
        )

        if np.isfinite(r):
            boot_rho.append(r)

    lower, upper = percentile_ci(
        boot_rho,
        BOOTSTRAP_ALPHA,
    )

    return rho, lower, upper

# Paired stratified bootstrap for AUROC and AURC
def bootstrap_rq4(
    df,
    signals,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):     

    point = {}

    for signal in signals:
        point[signal] = {
            "auroc": safe_auroc(
                df["answer_error"],
                df[signal],
            ),
            "aurc": aurc_score(df, signal),
        }

    boot = {
        signal: {"auroc": [], "aurc": []}
        for signal in signals
    }

    comparisons = {
        "combined_minus_evidence": {
            "auroc": [],
            "aurc": [],
        },
        "combined_minus_answer": {
            "auroc": [],
            "aurc": [],
        },
    }

    rng = np.random.default_rng(seed)

    for _ in range(n_bootstrap):
        sample = stratified_sample(df, rng)

        results = {}

        for signal in signals:
            auroc = safe_auroc(
                sample["answer_error"],
                sample[signal],
            )
            aurc = aurc_score(sample, signal)

            results[signal] = {
                "auroc": auroc,
                "aurc": aurc,
            }

            boot[signal]["auroc"].append(auroc)
            boot[signal]["aurc"].append(aurc)

        # Compare combined uncertainty with evidence uncertainty
        comparisons["combined_minus_evidence"]["auroc"].append(
            results["combined_uncertainty"]["auroc"]
            - results["evidence_uncertainty"]["auroc"]
        )

        comparisons["combined_minus_evidence"]["aurc"].append(
            results["combined_uncertainty"]["aurc"]
            - results["evidence_uncertainty"]["aurc"]
        )

        # Compare combined uncertainty with answer uncertainty
        comparisons["combined_minus_answer"]["auroc"].append(
            results["combined_uncertainty"]["auroc"]
            - results["answer_uncertainty"]["auroc"]
        )

        comparisons["combined_minus_answer"]["aurc"].append(
            results["combined_uncertainty"]["aurc"]
            - results["answer_uncertainty"]["aurc"]
        )

    metric_rows = []

    for signal in signals:
        for metric in ["auroc", "aurc"]:
            lower, upper = percentile_ci(
                boot[signal][metric],
                BOOTSTRAP_ALPHA,
            )

            metric_rows.append({
                "signal": signal,
                "metric": metric,
                "point_estimate": point[signal][metric],
                "ci_lower": lower,
                "ci_upper": upper,
            })

    comparison_rows = []

    comparison_pairs = {
        "combined_minus_evidence":
            ("combined_uncertainty", "evidence_uncertainty"),
        "combined_minus_answer":
            ("combined_uncertainty", "answer_uncertainty"),
    }

    for name, (combined, other) in comparison_pairs.items():
        for metric in ["auroc", "aurc"]:
            lower, upper = percentile_ci(
                comparisons[name][metric],
                BOOTSTRAP_ALPHA,
            )

            comparison_rows.append({
                "comparison": name,
                "metric": metric,
                "point_difference":
                    point[combined][metric]
                    - point[other][metric],
                "ci_lower": lower,
                "ci_upper": upper,
            })

    return metric_rows, comparison_rows


if RUN_BOOTSTRAP:

    bootstrap_metrics = []
    bootstrap_comparisons = []

    # Run bootstrap separately for each analysis dataset
    for i, (analysis_set, group) in enumerate(
        analysis.groupby("analysis_set", sort=False)
    ):
        group = group.reset_index(drop=True)
        seed = BOOTSTRAP_SEED + i

        # RQ3: relationship between evidence and answer uncertainty
        rho, rho_lower, rho_upper = bootstrap_spearman(
            group,
            seed=seed,
        )

        bootstrap_metrics.append({
            "analysis_set": analysis_set,
            "signal": "evidence_vs_answer_uncertainty",
            "metric": "spearman_rho",
            "point_estimate": rho,
            "ci_lower": rho_lower,
            "ci_upper": rho_upper,
        })

        # RQ4 requires both correct and incorrect cases
        if group["answer_error"].nunique() >= 2:

            metric_rows, comparison_rows = bootstrap_rq4(
                group,
                primary_signals,
                seed=seed + 1,
            )

            for row in metric_rows:
                row["analysis_set"] = analysis_set
                bootstrap_metrics.append(row)

            for row in comparison_rows:
                row["analysis_set"] = analysis_set
                bootstrap_comparisons.append(row)

    bootstrap_metrics = pd.DataFrame(bootstrap_metrics)
    bootstrap_comparisons = pd.DataFrame(
        bootstrap_comparisons
    )

    # Save bootstrap results
    bootstrap_metrics.to_csv(
        BOOTSTRAP_METRICS_PATH,
        index=False,
    )

    bootstrap_comparisons.to_csv(
        BOOTSTRAP_COMPARISONS_PATH,
        index=False,
    )

    display(bootstrap_metrics)
    display(bootstrap_comparisons)

else:
    print("Bootstrap skipped in SAMPLE_MODE.")

## 8. Reviews

#### 1) Manual Review of Judge Outputs

Manual review to compare a subset of the correctness judgements by LLM with manual assessment.

For ArchEHR-QA development cases, all 20  were included and for BioASQ, 10 cases
judged correct and 10 cases judged incorrect were randomly selected.

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

REVIEW_SEED = 42
REVIEW_PATH = ANALYSIS_DIR / "manual_judge_review.csv"

if REVIEW_PATH.exists():
    print("Manual review file already exists:")
    print(REVIEW_PATH)

else:
    # All 20 ArchEHR development cases
    arch_review = analysis[
        analysis["analysis_set"] == "archehr_train"
    ].copy()

    assert len(arch_review) == 20

    # BioASQ development cases
    bio = analysis[
        analysis["analysis_set"] == "bioasq_train"
    ].copy()

    # 10 judged correct cases
    bio_correct = bio[
        bio["judge_correct"] == 1
    ].sample(
        n=10,
        random_state=REVIEW_SEED,
    )

    # 10 judged incorrect cases
    bio_incorrect = bio[
        bio["judge_correct"] == 0
    ].sample(
        n=10,
        random_state=REVIEW_SEED,
    )

    # Combine review cases
    review = pd.concat(
        [
            arch_review,
            bio_correct,
            bio_incorrect,
        ],
        ignore_index=True,
    )

    # Shuffle review order
    review = review.sample(
        frac=1,
        random_state=REVIEW_SEED,
    ).reset_index(drop=True)

    review["review_id"] = [
        f"R{i:02d}"
        for i in range(1, len(review) + 1)
    ]

    # Add question and reference answer
    questions = []
    references = []

    for row in review.itertuples():

        case = case_by_key[
            (row.analysis_set, row.case_id)
        ]

        questions.append(
            case["question"]
        )

        references.append(
            "\n\n".join(
                case["reference_answers"]
            )
        )

    review["question"] = questions
    review["reference_answers"] = references

    # Keep only information needed for blinded review
    manual_review = review[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "question",
            "reference_answers",
            "representative_answer",
        ]
    ].copy()

    manual_review["manual_label"] = ""

    manual_review.to_csv(
        REVIEW_PATH,
        index=False,
    )

    print("Saved:", REVIEW_PATH)

In [ ]:
# Compare manual review with LLM judge result

manual_review = pd.read_csv(REVIEW_PATH)

manual_review["manual_label"] = (
    manual_review["manual_label"]
    .fillna("")
    .str.strip()
)

# Check manual labels
allowed_labels = {
    "Correct",
    "Incorrect",
    "Unclear",
}

invalid = manual_review[
    ~manual_review["manual_label"].isin(allowed_labels)
]

if len(invalid) > 0:
    display(
        invalid[
            [
                "review_id",
                "manual_label",
            ]
        ]
    )

    raise ValueError(
        "Use only Correct, Incorrect, or Unclear."
    )

# Add LLM judge result
judge_review_results = analysis[
    [
        "analysis_set",
        "case_id",
        "representative_answer",
        "judge_correct",
        "judge_reason",
    ]
].copy()

comparison = manual_review.merge(
    judge_review_results,
    on=[
        "analysis_set",
        "case_id",
        "representative_answer",
    ],
    how="left",
    validate="one_to_one",
)

comparison["judge_label"] = (
    comparison["judge_correct"]
    .astype(int)
    .map({
        1: "Correct",
        0: "Incorrect",
    })
)

# Compare manual and judge labels
comparison["comparison"] = [
    "Manual unclear"
    if manual == "Unclear"
    else "Agree"
    if manual == judge
    else "Disagree"
    for manual, judge in zip(
        comparison["manual_label"],
        comparison["judge_label"],
    )
]

print(
    comparison["comparison"].value_counts()
)

display(
    comparison.groupby(
        [
            "analysis_set",
            "comparison",
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)

# Review disagreements
disagreements = comparison[
    comparison["comparison"] == "Disagree"
].copy()

print(
    "Disagreements:",
    len(disagreements),
)

display(
    disagreements[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "manual_label",
            "judge_label",
            "judge_reason",
        ]
    ]
)

In [ ]:
# Review disagreement cases

disagreement_review = disagreements.copy()

questions = []
references = []

for row in disagreement_review.itertuples():

    case = case_by_key[
        (row.analysis_set, row.case_id)
    ]

    questions.append(
        case["question"]
    )

    references.append(
        "\n\n".join(
            case["reference_answers"]
        )
    )

disagreement_review["question"] = questions
disagreement_review["reference_answers"] = references

display(
    disagreement_review[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "question",
            "reference_answers",
            "representative_answer",
            "manual_label",
            "judge_label",
            "judge_reason",
        ]
    ]
)

#### 2) Case Review of the Four Profiles

Exploratory case-level review to examine the evidence and answer patterns underlying the four profiles.

In [ ]:
heldout = pd.read_csv(HELDOUT_PATH)

profiles = [
    "Stable–Stable",
    "Stable–Variable",
    "Unstable–Stable",
    "Unstable–Variable",
]

selected_cases = []

for profile in profiles:
    group = heldout[
        heldout["joint_profile"] == profile
    ]

    sampled = group.sample(
        n=1,
        random_state=42,
    )

    selected_cases.append(sampled)

selected_cases = pd.concat(
    selected_cases,
    ignore_index=True,
)

display(
    selected_cases[
        [
            "case_id",
            "joint_profile",
            "evidence_uncertainty",
            "answer_uncertainty",
            "judge_correct",
        ]
    ]
)

In [ ]:
# Load the detailed held-out outputs
selected_ids = selected_cases["case_id"].tolist()

evidence_runs = pd.DataFrame(
    load_jsonl(HELDOUT_DIR / "evidence_runs.jsonl")
)

evidence_summary = pd.read_csv(
    HELDOUT_DIR / "evidence_summary.csv"
)

answer_runs = pd.read_csv(
    HELDOUT_DIR / "answer_runs_with_clusters.csv"
)

judge_results = pd.DataFrame(
    load_jsonl(HELDOUT_DIR / "judge_results.jsonl")
)

cases = load_jsonl(
    PROCESSED_DIR / "archehr_test_cases.jsonl"
)

case_lookup = {
    case["case_id"]: case
    for case in cases
}

In [ ]:
for case_id in selected_ids:

    case = case_lookup[case_id]

    print("=="*5)
    print(case_id)
    print("=="*5)

    print("\n[QUESTION]")
    print(case["question"])

    print("\n[REFERENCE ANSWER]")
    print(case["reference_answers"][0])

    print("\n[EVIDENCE SELECTION RUNS]")
    display(
        evidence_runs[
            evidence_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "selected_sentence_ids",
            ]
        ]
    )

    summary = evidence_summary[
        evidence_summary["case_id"] == case_id
    ].iloc[0]

    representative_ids = json.loads(
        summary["selected_sentence_ids"]
    )

    print("\n[REPRESENTATIVE EVIDENCE]")
    for sentence in case["sentences"]:
        if str(sentence["sentence_id"]) in set(
            map(str, representative_ids)
        ):
            print(
                f'[{sentence["sentence_id"]}] '
                f'{sentence["text"]}'
            )

    print("\n[ANSWER RUNS]")
    display(
        answer_runs[
            answer_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "cluster_label",
                "answer",
            ]
        ]
    )

    print("\n[LLM JUDGE]")
    display(
        judge_results[
            judge_results["case_id"] == case_id
        ][
            [
                "correct",
                "representative_answer",
                "reason",
            ]
        ]
    )

#### 3) Evidence-Selection and Answer-Generation Variability in Relation to Answer Error

Exploratory review to examine case-level patterns of evidence-selection and answer-generation variability in relation to answer error.

In [ ]:
# Define the six groups and randomly sample three cases from each
heldout = pd.read_csv(HELDOUT_PATH)

review_groups = [
    ("Unstable–Variable incorrect", "Unstable–Variable", 0),
    ("Stable–Stable incorrect", "Stable–Stable", 0),
    ("Unstable–Stable incorrect", "Unstable–Stable", 0),
    ("Stable–Variable incorrect", "Stable–Variable", 0),
    ("Unstable–Stable correct", "Unstable–Stable", 1),
    ("Stable–Variable correct", "Stable–Variable", 1),
]

selected_review_cases = []

for group_name, profile, correct in review_groups:

    group = heldout[
        (heldout["joint_profile"] == profile)
        & (heldout["judge_correct"] == correct)
    ]

    sampled = group.sample(
        n=3,
        random_state=42,
    ).copy()

    sampled["review_group"] = group_name

    selected_review_cases.append(sampled)

selected_review_cases = pd.concat(
    selected_review_cases,
    ignore_index=True,
)

assert len(selected_review_cases) == 18

display(
    selected_review_cases[
        [
            "case_id",
            "review_group",
            "joint_profile",
            "evidence_uncertainty",
            "answer_uncertainty",
            "judge_correct",
        ]
    ]
)

In [ ]:
# Review the selected 18 cases

selected_ids = selected_review_cases[
    "case_id"
].tolist()

for case_id in selected_ids:

    case = case_lookup[case_id]

    print("=="*5)
    print(case_id)
    print("=="*5)

    print("\n[QUESTION]")
    print(case["question"])

    print("\n[REFERENCE ANSWER]")
    print(case["reference_answers"][0])

    print("\n[EVIDENCE SELECTION RUNS]")
    display(
        evidence_runs[
            evidence_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "selected_sentence_ids",
            ]
        ]
    )

    summary = evidence_summary[
        evidence_summary["case_id"] == case_id
    ].iloc[0]

    representative_ids = json.loads(
        summary["selected_sentence_ids"]
    )

    print("\n[REPRESENTATIVE EVIDENCE]")
    for sentence in case["sentences"]:
        if str(sentence["sentence_id"]) in set(
            map(str, representative_ids)
        ):
            print(
                f'[{sentence["sentence_id"]}] '
                f'{sentence["text"]}'
            )

    print("\n[ANSWER RUNS]")
    display(
        answer_runs[
            answer_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "cluster_label",
                "answer",
            ]
        ]
    )

    print("\n[LLM JUDGE]")
    display(
        judge_results[
            judge_results["case_id"] == case_id
        ][
            [
                "correct",
                "representative_answer",
                "reason",
            ]
        ]
    )

In [ ]:
# Check evidence changes in Unstable-Stable cases

for case_id in [
    "archehr_21",
    "archehr_43",
    "archehr_34",
    "archehr_74",
]:

    case = case_lookup[case_id]

    case_evidence = evidence_runs[
        evidence_runs["case_id"] == case_id
    ]

    print("\n", "=="*5)
    print(case_id)
    print("=="*5)

    all_ids = [
        str(x)
        for ids in case_evidence["selected_sentence_ids"]
        for x in ids
    ]

    counts = pd.Series(all_ids).value_counts()

    for sentence in case["sentences"]:

        sentence_id = str(
            sentence["sentence_id"]
        )

        if sentence_id in counts.index:
            print(
                f'[{sentence["sentence_id"]}] '
                f'{counts[sentence_id]}/10 - '
                f'{sentence["text"]}'
            )

In [ ]:
# Check evidence changes in Stable-Variable cases

for case_id in [
    "archehr_42",
    "archehr_28",
    "archehr_26",
    "archehr_95",
]:

    case = case_lookup[case_id]

    case_evidence = evidence_runs[
        evidence_runs["case_id"] == case_id
    ]

    print("\n", "=="*5)
    print(case_id)
    print("=="*5)

    all_ids = [
        str(x)
        for ids in case_evidence["selected_sentence_ids"]
        for x in ids
    ]

    counts = pd.Series(all_ids).value_counts()

    for sentence in case["sentences"]:

        sentence_id = str(
            sentence["sentence_id"]
        )

        if sentence_id in counts.index:
            print(
                f'[{sentence["sentence_id"]}] '
                f'{counts[sentence_id]}/10 - '
                f'{sentence["text"]}'
            )

In [ ]:
# Check evidence changes in Unstable-Variable cases

for case_id in [
    "archehr_51",
    "archehr_54",
    "archehr_82",
    "archehr_101",
]:

    case = case_lookup[case_id]

    case_evidence = evidence_runs[
        evidence_runs["case_id"] == case_id
    ]

    print("\n", "=="*5)
    print(case_id)
    print("=="*5)

    all_ids = [
        str(x)
        for ids in case_evidence["selected_sentence_ids"]
        for x in ids
    ]

    counts = pd.Series(all_ids).value_counts()

    for sentence in case["sentences"]:

        sentence_id = str(
            sentence["sentence_id"]
        )

        if sentence_id in counts.index:
            print(
                f'[{sentence["sentence_id"]}] '
                f'{counts[sentence_id]}/10 - '
                f'{sentence["text"]}'
            )